### Exploring Data

In [1]:
import os
data_dir = '/kaggle/input/datasets/mahmoudreda55/satellite-image-classification/data'

classes = [c for c in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, c))]
print(f"Classes found: {classes}")
for cls in classes:
    path = os.path.join(data_dir, cls)
    count = len([f for f in os.listdir(path) if os.path.isfile(os.path.join(path, f))])
    print(f"{cls}: {count} images")


Classes found: ['cloudy', 'desert', 'green_area', 'water']
cloudy: 1500 images
desert: 1131 images
green_area: 1500 images
water: 1500 images


### Splitting data

In [2]:
import random
import numpy as np
from PIL import Image
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

classes = [c for c in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, c))]
# subset_size = 500
img_size = (224,224)
images = []
labels = []

for cls in classes:
    class_path = os.path.join(data_dir, cls)
    all_images = [f for f in os.listdir(class_path) if os.path.isfile(os.path.join(class_path, f))]

    # selected_images = random.sample(all_images, min(subset_size, len(all_images)))

    for img_name in all_images:
        img_path = os.path.join(class_path, img_name)

        # load and resize
        img = Image.open(img_path).convert('RGB')
        img = img.resize(img_size)

      
        img_array = np.array(img, dtype=np.float32)        
        images.append(img_array)
        labels.append(cls)

X = np.array(images, dtype=np.float16)
y = np.array(labels)
print("X shape:", X.shape)
print("y shape:", y.shape)
le = LabelEncoder()
y_encoded = le.fit_transform(y)
y_cnn = to_categorical(y_encoded, num_classes=len(classes))


2026-05-19 13:50:44.385288: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779198644.738838      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779198644.836872      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779198645.729254      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779198645.729308      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779198645.729311      57 computation_placer.cc:177] computation placer alr

X shape: (5631, 224, 224, 3)
y shape: (5631,)


### Train Validation Split and Data Augmentation

In [3]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator

X_train, X_val, y_train, y_val = train_test_split(
    X, y_cnn, test_size=0.2, stratify=y_encoded, random_state=42
)
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(
    rescale=1./255
)


train_gen = train_datagen.flow(X_train, y_train, batch_size=32)
val_gen = val_datagen.flow(X_val, y_val, batch_size=32, shuffle=False)

### Building Custom CNN Model

In [4]:
import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Conv2D, BatchNormalization, ReLU, Add,
                                     GlobalAveragePooling2D, Dense,Dropout)
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
from sklearn.utils import shuffle

from tensorflow.keras.layers import MaxPooling2D


# Residual block
def residual_block(x, filters, kernel_size=3):
    shortcut = x
    x = Conv2D(filters, kernel_size, padding='same')(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    x = Conv2D(filters, kernel_size, padding='same')(x)
    x = BatchNormalization()(x)
    x= ReLU()(x)
    if shortcut.shape[-1] != filters:
        shortcut = Conv2D(filters, 1, padding='same')(shortcut)

    x = Add()([shortcut, x])
    x = ReLU()(x)
    return x

# CustomLandNet model

# Optimized for 224x224 - This is actually more "standard"
inputs = Input(shape=(224, 224, 3))

x = Conv2D(32, 3, padding='same')(inputs)
x = BatchNormalization()(x)
x = ReLU()(x)
x = MaxPooling2D(2)(x) # Now the image is 112x112

x = Conv2D(64, 3, padding='same')(x)
x = BatchNormalization()(x)
x = ReLU()(x)
x = MaxPooling2D(2)(x) # Now the image is 56x56

x = residual_block(x, 64)

x = GlobalAveragePooling2D()(x) # Averages 3,136 pixels instead of 50k!
x = Dropout(0.5)(x)
outputs = Dense(len(classes), activation='softmax')(x)

I0000 00:00:1779198729.520028      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1779198729.526622      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [5]:
model = Model(inputs, outputs)
model.compile(optimizer=Adam(learning_rate=0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 224, 224,  │        896 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 224, 224,  │        128 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu (ReLU)        │ (None, 224, 224,  │          0 │ batch_normalizat… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 112, 112,  │          0 │ re_lu[0][0]       │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 112, 112,  │     18,496 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 112, 112,  │        256 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_1 (ReLU)      │ (None, 112, 112,  │          0 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 56, 56,    │          0 │ re_lu_1[0][0]     │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 56, 56,    │     36,928 │ max_pooling2d_1[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 56, 56,    │        256 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_2 (ReLU)      │ (None, 56, 56,    │          0 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 56, 56,    │     36,928 │ re_lu_2[0][0]     │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 56, 56,    │        256 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_3 (ReLU)      │ (None, 56, 56,    │          0 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 56, 56,    │          0 │ max_pooling2d_1[… │
│                     │ 64)               │            │ re_lu_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_4 (ReLU)      │ (None, 56, 56,    │          0 │ add[0][0]       

 Total params: 94,404 (368.77 KB)

 Trainable params: 93,956 (367.02 KB)

 Non-trainable params: 448 (1.75 KB)

### Training the Custom CNN Model

In [6]:
from tensorflow.keras.callbacks import EarlyStopping

# 1. Define the EarlyStopping callback
early_stop = EarlyStopping(
    monitor='val_loss',     # Metric to monitor
    patience=5,             # Number of epochs to wait for improvement before stopping
    restore_best_weights=True # Keeps the weights from the best epoch
)

# 2. Pass it into the callbacks list in model.fit
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=30,
    callbacks=[early_stop]
)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/30


I0000 00:00:1779198735.066428     127 service.cc:152] XLA service 0x794c9c1080f0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1779198735.066464     127 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1779198735.066467     127 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1779198735.754976     127 cuda_dnn.cc:529] Loaded cuDNN version 91002


  2/141 ━━━━━━━━━━━━━━━━━━━━ 11s 86ms/step - accuracy: 0.4375 - loss: 1.4597 

I0000 00:00:1779198741.961097     127 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


141/141 ━━━━━━━━━━━━━━━━━━━━ 58s 342ms/step - accuracy: 0.7590 - loss: 0.6600 - val_accuracy: 0.4534 - val_loss: 2.2383
Epoch 2/30
141/141 ━━━━━━━━━━━━━━━━━━━━ 43s 306ms/step - accuracy: 0.8386 - loss: 0.4189 - val_accuracy: 0.5093 - val_loss: 2.0720
Epoch 3/30
141/141 ━━━━━━━━━━━━━━━━━━━━ 43s 307ms/step - accuracy: 0.8608 - loss: 0.3593 - val_accuracy: 0.6247 - val_loss: 0.8478
Epoch 4/30
141/141 ━━━━━━━━━━━━━━━━━━━━ 45s 316ms/step - accuracy: 0.8580 - loss: 0.3416 - val_accuracy: 0.7365 - val_loss: 0.5415
Epoch 5/30
141/141 ━━━━━━━━━━━━━━━━━━━━ 44s 311ms/step - accuracy: 0.8732 - loss: 0.3146 - val_accuracy: 0.8598 - val_loss: 0.4169
Epoch 6/30
141/141 ━━━━━━━━━━━━━━━━━━━━ 43s 307ms/step - accuracy: 0.8728 - loss: 0.3089 - val_accuracy: 0.8039 - val_loss: 0.4410
Epoch 7/30
141/141 ━━━━━━━━━━━━━━━━━━━━ 44s 308ms/step - accuracy: 0.8915 - loss: 0.2821 - val_accuracy: 0.6122 - val_loss: 1.0216
Epoch 8/30
141/141 ━━━━━━━━━━━━━━━━━━━━ 43s 308ms/step - accuracy: 0.8790 - loss: 0.2936 - val

### Saving Custom CNN Weights

In [7]:
# Save weights
model.save_weights("custom_landnet.weights.h5")
print("Weights saved to custom_landnet.h5")

Weights saved to custom_landnet.h5


### Preparing ResNet50 Data Generators

In [8]:
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.regularizers import l2


train_datagen_resnet = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)

val_datagen_resnet = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_gen_resnet = train_datagen_resnet.flow(
    X_train,
    y_train,
    batch_size=32,
    shuffle=True
)

val_gen_resnet = val_datagen_resnet.flow(
    X_val,
    y_val,
    batch_size=32,
    shuffle=False
)

### Building ResNet50 Transfer Learning Model

In [9]:
def build_resnet_model(num_classes):

    base_model = ResNet50(
        weights='imagenet',
        include_top=False,
        input_shape=(224, 224, 3)
    )

    # Freeze pretrained layers
    base_model.trainable = False

    inputs = Input(shape=(224, 224, 3))

    x = base_model(inputs, training=False)

    x = GlobalAveragePooling2D()(x)

    x = Dense(
        128,
        activation='relu',
        kernel_regularizer=l2(1e-4)
    )(x)

    x = Dropout(0.5)(x)

    outputs = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs, outputs)

    model.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

### Training ResNet50 Model

In [10]:
resnet_model = build_resnet_model(len(classes))

early_stop_resnet = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history_resnet = resnet_model.fit(
    train_gen_resnet,
    validation_data=val_gen_resnet,
    epochs=10,
    callbacks=[early_stop_resnet]
)

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
Epoch 1/10
141/141 ━━━━━━━━━━━━━━━━━━━━ 70s 408ms/step - accuracy: 0.8823 - loss: 0.3326 - val_accuracy: 0.9920 - val_loss: 0.0464
Epoch 2/10
141/141 ━━━━━━━━━━━━━━━━━━━━ 46s 327ms/step - accuracy: 0.9918 - loss: 0.0487 - val_accuracy: 0.9938 - val_loss: 0.0385
Epoch 3/10
141/141 ━━━━━━━━━━━━━━━━━━━━ 47s 334ms/step - accuracy: 0.9954 - loss: 0.0344 - val_accuracy: 0.9947 - val_loss: 0.0356
Epoch 4/10
141/141 ━━━━━━━━━━━━━━━━━━━━ 48s 338ms/step - accuracy: 0.9956 - loss: 0.0340 - val_accuracy: 0.9938 - val_loss: 0.0334
Epoch 5/10
141/141 ━━━━━━━━━━━━━━━━━━━━ 46s 326ms/step - accuracy: 0.9972 - loss: 0.0251 - val_accuracy: 0.9973 - val_loss: 0.0261
Epoch 6/10
141/141 ━━━━━━━━━━━━━━━━━━━━ 47s 329ms/step - accuracy: 0.9963 - loss: 0.0261 - val_accuracy: 0.9991 - val_loss: 0.0225
Epoch 7/10
141/141 ━━━━━━━━━━━━━━━━━━━━ 46s 329ms/step - accuracy: 0.9981 - loss: 0.0211 - val_accuracy: 0.9965 - val_loss: 0.0264
Epoch 8/10
141/141 ━━━━━━━━━━━━━

### Evaluating ResNet50 Model

In [11]:
resnet_loss, resnet_acc = resnet_model.evaluate(val_gen_resnet)

print(f"ResNet50 Validation Accuracy: {resnet_acc:.4f}")
print(f"ResNet50 Validation Loss: {resnet_loss:.4f}")

36/36 ━━━━━━━━━━━━━━━━━━━━ 3s 83ms/step - accuracy: 0.9998 - loss: 0.0165
ResNet50 Validation Accuracy: 0.9991
ResNet50 Validation Loss: 0.0198


### Model Comparison

In [12]:
cnn_acc = max(history.history['val_accuracy'])
resnet_acc_best = max(history_resnet.history['val_accuracy'])

print("\n========== MODEL COMPARISON ==========")
print(f"Custom CNN Best Accuracy : {cnn_acc:.4f}")
print(f"ResNet50 Best Accuracy   : {resnet_acc_best:.4f}")


========== MODEL COMPARISON ==========
Custom CNN Best Accuracy : 0.9130
ResNet50 Best Accuracy   : 0.9991
